# 05 — ConvLSTM model development and frozen test predictions

Full validation-stage ConvLSTM development followed by freezing the selected
model and generating held-out test predictions.

Final architecture: 21-day input, one ConvLSTM layer, 8 hidden channels,
3×3 kernel, 1×1 output convolution, Softplus output, no static LT/mask input.

The development run used PyTorch 2.8.0 and an NVIDIA RTX 3060 Laptop GPU.

## 0. Imports, reproducibility, GPU, and file configuration

This notebook starts from the **saved final harmonised catalogue**. It does not repeat USGS downloading, magnitude harmonisation, EDA, Poisson fitting, or XGBoost tuning.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("Device:", device)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
AUDIT_DIR = OUTPUT_DIR / "audit"

for _d in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR, PRED_DIR, AUDIT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

GRID_MASK_FILE = (
    DATA_DIR / "california_grid_centre_mask.csv"
)

FINAL_MW25_FILE = (
    DATA_DIR
    / "earthquake_california_grid_2010_2025_Mw25_centre_mask.csv"
)

if not GRID_MASK_FILE.exists():
    raise FileNotFoundError(
        f"Missing {GRID_MASK_FILE}. Put the centre-based grid mask "
        "in the same folder as this notebook, or update DATA_DIR."
    )

if not FINAL_MW25_FILE.exists():
    raise FileNotFoundError(
        f"Missing {FINAL_MW25_FILE}. This notebook requires the FINAL "
        "harmonised Mw-equivalent >=2.5 catalogue created by the main "
        "project notebook. Do not replace it with the older reported-M>=2.5 file."
    )

## 1. Load the final modelling catalogue and California centre mask

In [ ]:
START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

N_LAT = 53
N_LON = 55

INPUT_MAG = 2.5
TARGET_MAG = 3.0
FORECAST_HORIZON = 7
MAX_HISTORY_LENGTH = 30

grid_mask = pd.read_csv(GRID_MASK_FILE)
df = pd.read_csv(FINAL_MW25_FILE)

required_mask_cols = {
    "cell_id",
    "lat_idx",
    "lon_idx",
    "is_california"
}

missing_mask_cols = (
    required_mask_cols - set(grid_mask.columns)
)

if missing_mask_cols:
    raise ValueError(
        f"Grid mask is missing columns: {sorted(missing_mask_cols)}"
    )

required_catalogue_cols = {
    "time",
    "cell_id",
    "mag_Mw"
}

missing_catalogue_cols = (
    required_catalogue_cols - set(df.columns)
)

if missing_catalogue_cols:
    raise ValueError(
        f"Final catalogue is missing columns: "
        f"{sorted(missing_catalogue_cols)}"
    )

assert len(grid_mask) == 1080
assert grid_mask["cell_id"].is_unique
assert grid_mask["is_california"].fillna(False).all()

df["time"] = pd.to_datetime(
    df["time"],
    format="mixed",
    utc=True
)

df["date"] = (
    df["time"]
    .dt.floor("D")
    .dt.tz_localize(None)
)

dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

cell_ids = np.sort(
    grid_mask["cell_id"]
    .astype(int)
    .unique()
)

print("Final catalogue events:", len(df))
print("Retained California cells:", len(cell_ids))
print("Complete calendar days:", len(dates))
print(
    "Catalogue date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)
print(
    "Magnitude range (Mw-equivalent):",
    df["mag_Mw"].min(),
    "to",
    df["mag_Mw"].max()
)

assert len(df) == 30347
assert len(cell_ids) == 1080
assert len(dates) == 5844
assert (df["mag_Mw"] >= INPUT_MAG).all()

print("\nLoad checks passed.")

## 2. Construct daily input counts and daily target counts

The dynamic deep-learning input is the daily count of \(M_w^\ast\ge2.5\) earthquakes in each retained cell.

The forecasting target is the count of \(M_w^\ast\ge3.0\) earthquakes occurring over the **following seven days**.

In [ ]:
daily_counts_25 = (
    df.groupby(["date", "cell_id"])
      .size()
      .unstack(fill_value=0)
      .reindex(
          index=dates,
          columns=cell_ids,
          fill_value=0
      )
)

daily_counts_25.index.name = "date"

def build_daily_count_matrix(
    data,
    threshold,
    dates,
    cell_ids
):
    temp = data.loc[
        data["mag_Mw"] >= threshold
    ]

    counts = (
        temp.groupby(["date", "cell_id"])
            .size()
            .unstack(fill_value=0)
            .reindex(
                index=dates,
                columns=cell_ids,
                fill_value=0
            )
    )

    counts.index.name = "date"

    return counts

daily_counts_target = build_daily_count_matrix(
    data=df,
    threshold=TARGET_MAG,
    dates=dates,
    cell_ids=cell_ids
)

print("Input daily-count matrix:", daily_counts_25.shape)
print(
    "Total M_w* >= 2.5 events:",
    int(daily_counts_25.to_numpy().sum())
)

print("\nTarget daily-count matrix:", daily_counts_target.shape)
print(
    "Total M_w* >= 3.0 events:",
    int(daily_counts_target.to_numpy().sum())
)

assert daily_counts_25.shape == (5844, 1080)
assert daily_counts_target.shape == (5844, 1080)

assert daily_counts_25.to_numpy().sum() == 30347
assert daily_counts_target.to_numpy().sum() == 6586

print("\nDaily-count checks passed.")

## 3. Reconstruct the original \(53\times55\) spatial grid

The 1,080 retained cells remain the only effective modelling cells. The other locations in the rectangular tensor are zero-valued placeholders required by convolution.

In [ ]:
grid_lookup = (
    grid_mask
    .copy()
    .assign(cell_id=lambda x: x["cell_id"].astype(int))
    .sort_values("cell_id")
    .reset_index(drop=True)
)

assert np.array_equal(
    grid_lookup["cell_id"].to_numpy(),
    cell_ids
)

lat_idx = (
    grid_lookup["lat_idx"]
    .to_numpy(dtype=int)
)

lon_idx = (
    grid_lookup["lon_idx"]
    .to_numpy(dtype=int)
)

assert np.array_equal(
    lat_idx * N_LON + lon_idx,
    cell_ids
)

ca_mask = np.zeros(
    (N_LAT, N_LON),
    dtype=bool
)

ca_mask[lat_idx, lon_idx] = True

assert ca_mask.shape == (53, 55)
assert ca_mask.sum() == 1080

daily_maps_25 = np.zeros(
    (
        len(daily_counts_25),
        N_LAT,
        N_LON
    ),
    dtype=np.float32
)

daily_maps_25[:, lat_idx, lon_idx] = (
    daily_counts_25
    .to_numpy(dtype=np.float32)
)

reconstructed_input = (
    daily_maps_25[:, lat_idx, lon_idx]
)

assert np.count_nonzero(
    daily_maps_25[:, ~ca_mask]
) == 0

assert np.array_equal(
    reconstructed_input,
    daily_counts_25.to_numpy()
)

assert daily_maps_25.sum() == 30347

print("California mask:", ca_mask.shape)
print("Retained cells:", int(ca_mask.sum()))
print("Daily input maps:", daily_maps_25.shape)
print("Input-map event total:", daily_maps_25.sum())
print(
    "Non-zero values outside mask:",
    np.count_nonzero(daily_maps_25[:, ~ca_mask])
)
print("Exact reconstruction:", True)

## 4. Input-distribution diagnostic and log transformation

The daily day-cell counts are extremely sparse and heavy-tailed. The transform is applied to the **input only**:

\[
C'=\log(1+C).
\]

The target remains on its original count scale.

In [ ]:
x_counts = (
    daily_counts_25
    .to_numpy()
    .ravel()
)

nonzero_counts = x_counts[
    x_counts > 0
]

print("All day-cell observations:", len(x_counts))
print("Zero fraction:", np.mean(x_counts == 0))
print("Mean:", x_counts.mean())
print("Maximum:", x_counts.max())

print("\nNon-zero observations:", len(nonzero_counts))
print("Non-zero mean:", nonzero_counts.mean())
print("Non-zero median:", np.median(nonzero_counts))

print("\nNon-zero quantiles:")
for q in [
    0.50,
    0.90,
    0.95,
    0.99,
    0.999,
    1.0
]:
    print(
        f"{q:>5.3f}:",
        np.quantile(nonzero_counts, q)
    )

daily_maps_25_log = np.log1p(
    daily_maps_25
).astype(np.float32)

print("\nTransformed input shape:", daily_maps_25_log.shape)
print("Minimum:", daily_maps_25_log.min())
print("Maximum:", daily_maps_25_log.max())
print(
    "Non-zero outside CA:",
    np.count_nonzero(
        daily_maps_25_log[:, ~ca_mask]
    )
)
print(
    "Zero locations preserved:",
    np.array_equal(
        daily_maps_25 == 0,
        daily_maps_25_log == 0
    )
)

assert np.count_nonzero(
    daily_maps_25_log[:, ~ca_mask]
) == 0

assert np.array_equal(
    daily_maps_25 == 0,
    daily_maps_25_log == 0
)

## 5. Construct the future 7-day target maps

For forecast origin \(t\),

\[
Y^{(7)}_{t,g}
=
\sum_{h=1}^{7} C^{(3.0)}_{t+h,g}.
\]

Day \(t\) is not included in the target.

In [ ]:
future7_target = sum(
    daily_counts_target.shift(-h)
    for h in range(1, FORECAST_HORIZON + 1)
)

future7_target = (
    future7_target
    .iloc[:-FORECAST_HORIZON]
    .copy()
)

future7_target_maps = np.zeros(
    (
        len(future7_target),
        N_LAT,
        N_LON
    ),
    dtype=np.float32
)

future7_target_maps[:, lat_idx, lon_idx] = (
    future7_target
    .to_numpy(dtype=np.float32)
)

assert future7_target.shape == (5837, 1080)
assert future7_target_maps.shape == (5837, 53, 55)

assert np.count_nonzero(
    future7_target_maps[:, ~ca_mask]
) == 0

assert np.array_equal(
    future7_target_maps[:, lat_idx, lon_idx],
    future7_target.to_numpy()
)

print("Future-7 target matrix:", future7_target.shape)
print(
    "Forecast-origin range:",
    future7_target.index[0],
    "to",
    future7_target.index[-1]
)
print("Future-7 target maps:", future7_target_maps.shape)
print(
    "Non-zero outside CA:",
    np.count_nonzero(
        future7_target_maps[:, ~ca_mask]
    )
)
print("Exact target reconstruction:", True)

## 6. Define common forecast origins and chronological splits

All candidate history lengths will use the same forecast origins. Because the longest planned history length is 30 days, the first common origin is 2010-01-30.

The test indices are created for bookkeeping only. **Do not create test predictions or use test performance during model development.**

In [ ]:
input_dates = pd.DatetimeIndex(
    daily_counts_25.index
)

target_dates = pd.DatetimeIndex(
    future7_target.index
)

common_origins = pd.date_range(
    start="2010-01-30",
    end="2025-12-24",
    freq="D"
)

input_origin_pos = (
    input_dates.get_indexer(
        common_origins
    )
)

target_origin_pos = (
    target_dates.get_indexer(
        common_origins
    )
)

assert np.all(input_origin_pos >= 0)
assert np.all(target_origin_pos >= 0)

train_mask_dl = (
    (common_origins >= pd.Timestamp("2010-01-30")) &
    (common_origins <= pd.Timestamp("2019-12-24"))
)

val_mask_dl = (
    (common_origins >= pd.Timestamp("2020-01-01")) &
    (common_origins <= pd.Timestamp("2022-12-24"))
)

test_mask_dl = (
    (common_origins >= pd.Timestamp("2023-01-01")) &
    (common_origins <= pd.Timestamp("2025-12-24"))
)

train_indices_dl = np.flatnonzero(
    train_mask_dl
)

val_indices_dl = np.flatnonzero(
    val_mask_dl
)

test_indices_dl = np.flatnonzero(
    test_mask_dl
)

print("Common forecast origins:", len(common_origins))
print(
    "Origin range:",
    common_origins[0],
    "to",
    common_origins[-1]
)

print("\nFirst input position:", input_origin_pos[0])
print("First target position:", target_origin_pos[0])

print("\nTraining origins:", len(train_indices_dl))
print("Validation origins:", len(val_indices_dl))
print("Test origins:", len(test_indices_dl))

assert len(common_origins) == 5808
assert input_origin_pos[0] == 29
assert target_origin_pos[0] == 29

assert len(train_indices_dl) == 3616
assert len(val_indices_dl) == 1089
assert len(test_indices_dl) == 1089

print("\nTemporal alignment checks passed.")

## 7. Explicit pseudo-prospective alignment check

For \(L=30\), the first sample must use 2010-01-01 to 2010-01-30 as input and 2010-01-31 to 2010-02-06 as its target.

In [ ]:
L_CHECK = 30
i = 0

origin = common_origins[i]
origin_pos = input_origin_pos[i]

input_start_pos = (
    origin_pos - L_CHECK + 1
)

input_end_pos = origin_pos

x_example = daily_maps_25[
    input_start_pos:input_end_pos + 1
]

y_example = future7_target_maps[
    target_origin_pos[i]
]

print("Forecast origin:", origin)
print(
    "Input dates:",
    input_dates[input_start_pos],
    "to",
    input_dates[input_end_pos]
)
print(
    "Target dates:",
    origin + pd.Timedelta(days=1),
    "to",
    origin + pd.Timedelta(days=7)
)

print("\nX shape:", x_example.shape)
print("Y shape:", y_example.shape)
print("Input total count:", x_example.sum())
print("Target total count:", y_example.sum())

assert x_example.shape == (30, 53, 55)
assert y_example.shape == (53, 55)

## 8. PyTorch sequence Dataset

The Dataset generates sequences dynamically, so the full \(N\times L\times53\times55\) tensor does not need to be stored in memory.

In [ ]:
class EarthquakeSequenceDataset(Dataset):
    # input_maps: (5844, 53, 55), log1p M_w* >= 2.5 daily maps
    # target_maps: (5837, 53, 55), future 7-day M_w* >= 3.0 maps
    # selected_indices: positions within common_origins
    # history_length: number of consecutive daily maps ending at t

    def __init__(
        self,
        input_maps,
        target_maps,
        input_origin_pos,
        target_origin_pos,
        selected_indices,
        history_length
    ):
        self.input_maps = input_maps
        self.target_maps = target_maps
        self.input_origin_pos = input_origin_pos
        self.target_origin_pos = target_origin_pos
        self.selected_indices = np.asarray(
            selected_indices,
            dtype=int
        )
        self.history_length = int(
            history_length
        )

    def __len__(self):
        return len(
            self.selected_indices
        )

    def __getitem__(self, idx):
        k = self.selected_indices[idx]

        origin_pos = (
            self.input_origin_pos[k]
        )

        start_pos = (
            origin_pos
            - self.history_length
            + 1
        )

        end_pos = origin_pos + 1

        x = np.ascontiguousarray(
            self.input_maps[
                start_pos:end_pos
            ],
            dtype=np.float32
        )

        y = np.ascontiguousarray(
            self.target_maps[
                self.target_origin_pos[k]
            ],
            dtype=np.float32
        )

        # (L,H,W) -> (L,C=1,H,W)
        x = (
            torch.from_numpy(x)
            .unsqueeze(1)
        )

        # (H,W) -> (C=1,H,W)
        y = (
            torch.from_numpy(y)
            .unsqueeze(0)
        )

        return x, y

## 9. California loss mask

The full rectangular tensor is used by convolution, but later the loss must be calculated only over the 1,080 retained California cells.

In [ ]:
ca_mask_tensor = (
    torch.from_numpy(
        ca_mask.astype(np.float32)
    )
    .unsqueeze(0)
    .unsqueeze(0)
)

print(
    "Mask tensor shape:",
    ca_mask_tensor.shape
)

print(
    "Retained cells:",
    ca_mask_tensor.sum().item()
)

assert ca_mask_tensor.shape == (1, 1, 53, 55)
assert ca_mask_tensor.sum().item() == 1080

## 10. Dataset / DataLoader smoke test

Use \(L=30\) here because it is the largest planned history length and therefore the most demanding case for memory.

This is only a data-pipeline check. It does not select \(L=30\) as the final model.

In [ ]:
HISTORY_LENGTH = 30
BATCH_SIZE = 8

train_dataset = EarthquakeSequenceDataset(
    input_maps=daily_maps_25_log,
    target_maps=future7_target_maps,
    input_origin_pos=input_origin_pos,
    target_origin_pos=target_origin_pos,
    selected_indices=train_indices_dl,
    history_length=HISTORY_LENGTH
)

val_dataset = EarthquakeSequenceDataset(
    input_maps=daily_maps_25_log,
    target_maps=future7_target_maps,
    input_origin_pos=input_origin_pos,
    target_origin_pos=target_origin_pos,
    selected_indices=val_indices_dl,
    history_length=HISTORY_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

x0, y0 = train_dataset[0]

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

print("\nSingle X:", x0.shape)
print("Single Y:", y0.shape)

xb, yb = next(
    iter(train_loader)
)

print("\nBatch X:", xb.shape)
print("Batch Y:", yb.shape)

assert len(train_dataset) == 3616
assert len(val_dataset) == 1089

assert x0.shape == (30, 1, 53, 55)
assert y0.shape == (1, 53, 55)

assert xb.shape[1:] == (30, 1, 53, 55)
assert yb.shape[1:] == (1, 53, 55)

print("\nDataLoader smoke test passed.")

## 11. Planned first modelling experiment

The first ConvLSTM experiment will compare a small set of history lengths while holding the reference architecture fixed:

\[
L\in\{7,21,30\}.
\]

- \(L=7\): one-week daily history
- \(L=21\): approximately three weeks
- \(L=30\): approximately one month

The same 3,616 training origins and 1,089 validation origins must be used for every value of \(L\).

### Next implementation step

Build a deliberately small reference ConvLSTM:

- input channels: 1
- one ConvLSTM layer
- hidden channels: 16
- kernel: \(3\times3\)
- one convolutional output head
- positive expected-count output
- masked Poisson negative log-likelihood over the 1,080 California cells

Do not add the long-term spatial map or tune architecture parameters until the basic history-length experiment is working.

## 12. Test-set guardrail

The test period is reserved for final out-of-sample evaluation.

During model development:

- use `train_indices_dl` to fit model weights;
- use `val_indices_dl` for history-length, architecture, and hyperparameter selection;
- do **not** use `test_indices_dl` to make modelling decisions.

A test Dataset/DataLoader should only be created after the deep-learning specification has been frozen.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class ConvLSTMCell(nn.Module):
    def __init__(
        self,
        input_channels,
        hidden_channels,
        kernel_size=3
    ):
        super().__init__()

        self.hidden_channels = hidden_channels

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            in_channels=input_channels + hidden_channels,
            out_channels=4 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding
        )

    def forward(self, x, h, c):

        combined = torch.cat(
            [x, h],
            dim=1
        )

        gates = self.conv(combined)

        i, f, o, g = torch.chunk(
            gates,
            chunks=4,
            dim=1
        )

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_next = (
            f * c
            + i * g
        )

        h_next = (
            o * torch.tanh(c_next)
        )

        return h_next, c_next

In [ ]:
class EarthquakeConvLSTM(nn.Module):
    def __init__(
        self,
        input_channels=1,
        hidden_channels=16,
        kernel_size=3
    ):
        super().__init__()

        self.hidden_channels = hidden_channels

        self.convlstm = ConvLSTMCell(
            input_channels=input_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size
        )

        # Convert final ConvLSTM hidden state
        # into one expected-count map
        self.output_head = nn.Conv2d(
            in_channels=hidden_channels,
            out_channels=1,
            kernel_size=1
        )

    def forward(self, x):

        # x:
        # (B, L, C, H, W)

        B, L, C, H, W = x.shape

        h = torch.zeros(
            B,
            self.hidden_channels,
            H,
            W,
            device=x.device,
            dtype=x.dtype
        )

        c = torch.zeros_like(h)

        # Process the daily maps sequentially
        for t in range(L):
            h, c = self.convlstm(
                x[:, t],
                h,
                c
            )

        raw_output = self.output_head(h)

        # Positive expected counts
        rate = F.softplus(raw_output) + 1e-8

        return rate

In [ ]:
model = EarthquakeConvLSTM(
    input_channels=1,
    hidden_channels=16,
    kernel_size=3
).to(device)

xb, yb = next(iter(train_loader))

xb = xb.to(
    device,
    non_blocking=True
)

yb = yb.to(
    device,
    non_blocking=True
)

with torch.no_grad():
    pred = model(xb)

print("Input:", xb.shape)
print("Target:", yb.shape)
print("Prediction:", pred.shape)

print(
    "Prediction minimum:",
    pred.min().item()
)

print(
    "Prediction maximum:",
    pred.max().item()
)

print(
    "All predictions finite:",
    torch.isfinite(pred).all().item()
)

print(
    "All predictions positive:",
    (pred > 0).all().item()
)

In [ ]:
n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    f"{n_params:,}"
)

In [ ]:
# ============================================================
# Training-period mean future-7 count per California cell
# ============================================================

train_target_maps = future7_target_maps[
    target_origin_pos[train_indices_dl]
]

train_target_values = train_target_maps[
    :, ca_mask
]

train_mean_target = train_target_values.mean()

print(
    "Training mean future-7 count per cell:",
    train_mean_target
)

print(
    "Equivalent statewide 7-day mean:",
    train_mean_target * ca_mask.sum()
)

In [ ]:
class EarthquakeConvLSTM(nn.Module):
    def __init__(
        self,
        input_channels=1,
        hidden_channels=16,
        kernel_size=3,
        initial_mean_rate=None
    ):
        super().__init__()

        self.hidden_channels = hidden_channels

        self.convlstm = ConvLSTMCell(
            input_channels=input_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size
        )

        self.output_head = nn.Conv2d(
            in_channels=hidden_channels,
            out_channels=1,
            kernel_size=1
        )

        # ----------------------------------------------------
        # Initialise output bias to training mean count
        # ----------------------------------------------------

        if initial_mean_rate is not None:

            mean_rate = float(initial_mean_rate)

            if mean_rate <= 0:
                raise ValueError(
                    "initial_mean_rate must be positive."
                )

            # inverse softplus:
            # b = log(exp(mean_rate) - 1)
            initial_bias = np.log(
                np.expm1(mean_rate)
            )

            nn.init.constant_(
                self.output_head.bias,
                initial_bias
            )

    def forward(self, x):

        B, L, C, H, W = x.shape

        h = torch.zeros(
            B,
            self.hidden_channels,
            H,
            W,
            device=x.device,
            dtype=x.dtype
        )

        c = torch.zeros_like(h)

        for t in range(L):
            h, c = self.convlstm(
                x[:, t],
                h,
                c
            )

        raw_output = self.output_head(h)

        rate = F.softplus(
            raw_output
        ) + 1e-8

        return rate

In [ ]:
model = EarthquakeConvLSTM(
    input_channels=1,
    hidden_channels=16,
    kernel_size=3,
    initial_mean_rate=train_mean_target
).to(device)

In [ ]:
xb, yb = next(iter(train_loader))

xb = xb.to(
    device,
    non_blocking=True
)

yb = yb.to(
    device,
    non_blocking=True
)

with torch.no_grad():
    pred = model(xb)

print(
    "Training mean target:",
    train_mean_target
)

print(
    "Prediction mean:",
    pred[:, :, ca_mask].mean().item()
)

print(
    "Prediction minimum:",
    pred[:, :, ca_mask].min().item()
)

print(
    "Prediction maximum:",
    pred[:, :, ca_mask].max().item()
)

In [ ]:
def masked_poisson_nll(
    rate,
    target,
    mask
):
    """
    Exact mean Poisson negative log-likelihood
    over retained California cells only.

    rate:
        (B, 1, H, W)

    target:
        (B, 1, H, W)

    mask:
        (1, 1, H, W)
    """

    rate = torch.clamp(
        rate,
        min=1e-8
    )

    nll = (
        rate
        - target * torch.log(rate)
        + torch.lgamma(target + 1.0)
    )

    valid_mask = mask.expand_as(
        target
    )

    loss = (
        nll * valid_mask
    ).sum() / valid_mask.sum()

    return loss

In [ ]:
mask_gpu = ca_mask_tensor.to(device)

with torch.no_grad():

    pred = model(xb)

    loss = masked_poisson_nll(
        rate=pred,
        target=yb,
        mask=mask_gpu
    )

print(
    "Initial masked Poisson NLL:",
    loss.item()
)

print(
    "Initial Poisson log score:",
    -loss.item()
)

In [ ]:
# ============================================================
# Full validation evaluation
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    mask,
    device
):
    model.eval()

    total_nll = 0.0
    total_valid = 0.0

    mask = mask.to(device)

    for xb, yb in data_loader:

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        rate = model(xb)

        rate = torch.clamp(
            rate,
            min=1e-8
        )

        nll = (
            rate
            - yb * torch.log(rate)
            + torch.lgamma(yb + 1.0)
        )

        valid_mask = mask.expand_as(yb)

        total_nll += (
            nll * valid_mask
        ).sum().item()

        total_valid += (
            valid_mask.sum().item()
        )

    mean_nll = (
        total_nll / total_valid
    )

    log_score = -mean_nll

    return mean_nll, log_score

In [ ]:
# ============================================================
# Train for one epoch
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    optimizer,
    mask,
    device
):
    model.train()

    total_nll = 0.0
    total_valid = 0.0

    mask = mask.to(device)

    for xb, yb in data_loader:

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        rate = model(xb)

        loss = masked_poisson_nll(
            rate=rate,
            target=yb,
            mask=mask
        )

        loss.backward()

        optimizer.step()

        # ----------------------------------------------------
        # Store exact weighted total for epoch reporting
        # ----------------------------------------------------

        with torch.no_grad():

            rate_eval = torch.clamp(
                rate,
                min=1e-8
            )

            nll = (
                rate_eval
                - yb * torch.log(rate_eval)
                + torch.lgamma(yb + 1.0)
            )

            valid_mask = mask.expand_as(yb)

            total_nll += (
                nll * valid_mask
            ).sum().item()

            total_valid += (
                valid_mask.sum().item()
            )

    mean_nll = (
        total_nll / total_valid
    )

    return mean_nll

In [ ]:
# ============================================================
# Training with early stopping
# ============================================================

import copy
import time


def fit_model(
    model,
    train_loader,
    val_loader,
    mask,
    device,
    learning_rate=1e-3,
    max_epochs=500,
    patience=20
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.99)
    )

    history = {
        "train_nll": [],
        "val_nll": [],
        "val_log_score": []
    }

    best_val_nll = np.inf
    best_epoch = None
    best_state = None

    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(
        1,
        max_epochs + 1
    ):

        train_nll = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            mask=mask,
            device=device
        )

        val_nll, val_log_score = evaluate_model(
            model=model,
            data_loader=val_loader,
            mask=mask,
            device=device
        )

        history["train_nll"].append(
            train_nll
        )

        history["val_nll"].append(
            val_nll
        )

        history["val_log_score"].append(
            val_log_score
        )

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if val_nll < best_val_nll:

            best_val_nll = val_nll
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if verbose:
            print(
            f"Epoch {epoch:3d} | "
            f"train NLL = {train_nll:.6f} | "
            f"val NLL = {val_nll:.6f} | "
            f"val log score = {val_log_score:.6f}")
        if (
            epochs_without_improvement
            >= patience
        ):

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break

    # --------------------------------------------------------
    # Restore best validation model
    # --------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    elapsed = (
        time.time() - start_time
    )

    print(
        f"\nBest epoch: {best_epoch}"
    )

    print(
        f"Best validation NLL: "
        f"{best_val_nll:.6f}"
    )

    print(
        f"Best validation log score: "
        f"{-best_val_nll:.6f}"
    )

    print(
        f"Training time: "
        f"{elapsed / 60:.2f} minutes"
    )

    return (
        model,
        history,
        best_epoch,
        best_val_nll
    )

In [ ]:
# ============================================================
# First reference training run: L = 30
# ============================================================

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_30 = EarthquakeConvLSTM(
    input_channels=1,
    hidden_channels=16,
    kernel_size=3,
    initial_mean_rate=train_mean_target
).to(device)

model_30, history_30, best_epoch_30, best_val_nll_30 = (
    fit_model(
        model=model_30,
        train_loader=train_loader,
        val_loader=val_loader,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=500,
        patience=20
    )
)

In [ ]:
def make_loaders_for_history(
    history_length,
    batch_size=8
):
    train_dataset = EarthquakeSequenceDataset(
        input_maps=daily_maps_25_log,
        target_maps=future7_target_maps,
        input_origin_pos=input_origin_pos,
        target_origin_pos=target_origin_pos,
        selected_indices=train_indices_dl,
        history_length=history_length
    )

    val_dataset = EarthquakeSequenceDataset(
        input_maps=daily_maps_25_log,
        target_maps=future7_target_maps,
        input_origin_pos=input_origin_pos,
        target_origin_pos=target_origin_pos,
        selected_indices=val_indices_dl,
        history_length=history_length
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader

In [ ]:
def run_history_experiment(
    history_length
):
    print(
        f"\n{'='*60}\n"
        f"History length = {history_length} days\n"
        f"{'='*60}"
    )

    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_loader_L, val_loader_L = (
        make_loaders_for_history(
            history_length=history_length,
            batch_size=8
        )
    )

    model_L = EarthquakeConvLSTM(
        input_channels=1,
        hidden_channels=16,
        kernel_size=3,
        initial_mean_rate=train_mean_target
    ).to(device)

    (
        model_L,
        history_L,
        best_epoch_L,
        best_val_nll_L
    ) = fit_model(
        model=model_L,
        train_loader=train_loader_L,
        val_loader=val_loader_L,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=500,
        patience=20
    )

    return {
        "history_length": history_length,
        "model": model_L,
        "history": history_L,
        "best_epoch": best_epoch_L,
        "best_val_nll": best_val_nll_L,
        "best_val_log_score": -best_val_nll_L
    }

In [ ]:
result_7 = run_history_experiment(7)

In [ ]:
result_21 = run_history_experiment(21)

In [ ]:
result_30 = {
    "history_length": 30,
    "model": model_30,
    "history": history_30,
    "best_epoch": best_epoch_30,
    "best_val_nll": best_val_nll_30,
    "best_val_log_score": -best_val_nll_30
}

In [ ]:
history_results = pd.DataFrame([
    {
        "History days": result_7["history_length"],
        "Best epoch": result_7["best_epoch"],
        "Validation NLL": result_7["best_val_nll"],
        "Validation log score":
            result_7["best_val_log_score"]
    },
    {
        "History days": result_21["history_length"],
        "Best epoch": result_21["best_epoch"],
        "Validation NLL": result_21["best_val_nll"],
        "Validation log score":
            result_21["best_val_log_score"]
    },
    {
        "History days": 30,
        "Best epoch": best_epoch_30,
        "Validation NLL": best_val_nll_30,
        "Validation log score":
            -best_val_nll_30
    }
])

history_results = (
    history_results
    .sort_values(
        "Validation log score",
        ascending=False
    )
    .reset_index(drop=True)
)

history_results

In [ ]:
def make_loaders_for_history(
    history_length,
    seed,
    batch_size=8
):
    train_dataset = EarthquakeSequenceDataset(
        input_maps=daily_maps_25_log,
        target_maps=future7_target_maps,
        input_origin_pos=input_origin_pos,
        target_origin_pos=target_origin_pos,
        selected_indices=train_indices_dl,
        history_length=history_length
    )

    val_dataset = EarthquakeSequenceDataset(
        input_maps=daily_maps_25_log,
        target_maps=future7_target_maps,
        input_origin_pos=input_origin_pos,
        target_origin_pos=target_origin_pos,
        selected_indices=val_indices_dl,
        history_length=history_length
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader

In [ ]:
def run_seed_experiment(
    history_length,
    seed
):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_loader_L, val_loader_L = (
        make_loaders_for_history(
            history_length=history_length,
            seed=seed,
            batch_size=8
        )
    )

    model_L = EarthquakeConvLSTM(
        input_channels=1,
        hidden_channels=16,
        kernel_size=3,
        initial_mean_rate=train_mean_target
    ).to(device)

    (
        model_L,
        history_L,
        best_epoch_L,
        best_val_nll_L
    ) = fit_model(
        model=model_L,
        train_loader=train_loader_L,
        val_loader=val_loader_L,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=500,
        patience=20,
        verbose=False
    )

    return {
        "History days": history_length,
        "Seed": seed,
        "Best epoch": best_epoch_L,
        "Validation NLL": best_val_nll_L,
        "Validation log score": -best_val_nll_L
    }

In [ ]:
# ============================================================
# Training with early stopping
# ============================================================

import copy
import time


def fit_model(
    model,
    train_loader,
    val_loader,
    mask,
    device,
    learning_rate=1e-3,
    max_epochs=500,
    patience=20,
    verbose=True
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.99)
    )

    history = {
        "train_nll": [],
        "val_nll": [],
        "val_log_score": []
    }

    best_val_nll = np.inf
    best_epoch = None
    best_state = None

    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(
        1,
        max_epochs + 1
    ):

        train_nll = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            mask=mask,
            device=device
        )

        val_nll, val_log_score = evaluate_model(
            model=model,
            data_loader=val_loader,
            mask=mask,
            device=device
        )

        history["train_nll"].append(
            train_nll
        )

        history["val_nll"].append(
            val_nll
        )

        history["val_log_score"].append(
            val_log_score
        )

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if val_nll < best_val_nll:

            best_val_nll = val_nll
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        # ----------------------------------------------------
        # Optional epoch-by-epoch output
        # ----------------------------------------------------

        if verbose:
            print(
                f"Epoch {epoch:3d} | "
                f"train NLL = {train_nll:.6f} | "
                f"val NLL = {val_nll:.6f} | "
                f"val log score = {val_log_score:.6f}"
            )

        if (
            epochs_without_improvement
            >= patience
        ):

            if verbose:
                print(
                    f"\nEarly stopping at epoch {epoch}."
                )

            break

    # --------------------------------------------------------
    # Restore best validation model
    # --------------------------------------------------------

    if best_state is None:
        raise RuntimeError(
            "No valid model state was saved."
        )

    model.load_state_dict(
        best_state
    )

    elapsed = (
        time.time() - start_time
    )

    if verbose:
        print(
            f"\nBest epoch: {best_epoch}"
        )

        print(
            f"Best validation NLL: "
            f"{best_val_nll:.6f}"
        )

        print(
            f"Best validation log score: "
            f"{-best_val_nll:.6f}"
        )

        print(
            f"Training time: "
            f"{elapsed / 60:.2f} minutes"
        )

    return (
        model,
        history,
        best_epoch,
        best_val_nll
    )

In [ ]:
seed_results = []

for L in [21, 30]:
    for seed in [2026, 2027, 2028]:

        print(
            f"Running L={L}, seed={seed}..."
        )

        result = run_seed_experiment(
            history_length=L,
            seed=seed
        )

        seed_results.append(result)

        print(
            f"Best epoch = {result['Best epoch']}, "
            f"log score = "
            f"{result['Validation log score']:.6f}"
        )

In [ ]:
results_14 = []

for seed in [2026, 2027, 2028]:

    print(
        f"Running L=14, seed={seed}..."
    )

    result = run_seed_experiment(
        history_length=14,
        seed=seed
    )

    results_14.append(result)

    print(
        f"Best epoch = {result['Best epoch']}, "
        f"log score = "
        f"{result['Validation log score']:.6f}"
    )

In [ ]:
results_7_seeds = []

for seed in [2026, 2027, 2028]:

    print(f"Running L=7, seed={seed}...")

    result = run_seed_experiment(
        history_length=7,
        seed=seed
    )

    results_7_seeds.append(result)

    print(
        f"Best epoch = {result['Best epoch']}, "
        f"log score = "
        f"{result['Validation log score']:.6f}"
    )

In [ ]:
import gc


def run_quick_history_check(
    history_length,
    seed=2026,
    max_epochs=30
):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_loader_L, val_loader_L = (
        make_loaders_for_history(
            history_length=history_length,
            seed=seed,
            batch_size=8
        )
    )

    model_L = EarthquakeConvLSTM(
        input_channels=1,
        hidden_channels=16,
        kernel_size=3,
        initial_mean_rate=train_mean_target
    ).to(device)

    (
        model_L,
        history_L,
        best_epoch_L,
        best_val_nll_L
    ) = fit_model(
        model=model_L,
        train_loader=train_loader_L,
        val_loader=val_loader_L,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=max_epochs,

        # effectively prevents early stopping
        # during this short screening run
        patience=max_epochs,

        verbose=False
    )

    result = {
        "History days": history_length,
        "Seed": seed,
        "Best epoch": best_epoch_L,
        "Validation NLL": best_val_nll_L,
        "Validation log score": -best_val_nll_L
    }

    del model_L
    del train_loader_L
    del val_loader_L

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [ ]:
quick_history_results = []

for L in [20, 21, 22]:

    print(f"Quick check: L={L}...")

    result = run_quick_history_check(
        history_length=L,
        seed=2026,
        max_epochs=30
    )

    quick_history_results.append(result)

    print(
        f"Best epoch = {result['Best epoch']}, "
        f"log score = "
        f"{result['Validation log score']:.6f}"
    )

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Make CUDA operations as deterministic as reasonably possible
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True
        )
    except Exception:
        pass

In [ ]:
set_seed(seed)

In [ ]:
generator = torch.Generator()
generator.manual_seed(seed)

In [ ]:
import random
import numpy as np
import torch


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True
        )
    except Exception:
        pass

In [ ]:
import random
import numpy as np
import torch


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True
        )
    except Exception:
        pass

In [ ]:
def run_capacity_experiment(
    hidden_channels,
    seed=2026
):
    set_seed(seed)

    train_loader_H, val_loader_H = (
        make_loaders_for_history(
            history_length=21,
            seed=seed,
            batch_size=8
        )
    )

    model_H = EarthquakeConvLSTM(
        input_channels=1,
        hidden_channels=hidden_channels,
        kernel_size=3,
        initial_mean_rate=train_mean_target
    ).to(device)

    n_params = sum(
        p.numel()
        for p in model_H.parameters()
        if p.requires_grad
    )

    (
        model_H,
        history_H,
        best_epoch_H,
        best_val_nll_H
    ) = fit_model(
        model=model_H,
        train_loader=train_loader_H,
        val_loader=val_loader_H,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=500,
        patience=20,
        verbose=False
    )

    return {
        "Hidden channels": hidden_channels,
        "Parameters": n_params,
        "Seed": seed,
        "Best epoch": best_epoch_H,
        "Validation NLL": best_val_nll_H,
        "Validation log score": -best_val_nll_H,
        "model": model_H,
        "history": history_H
    }

In [ ]:
capacity_results = []

for H in [8, 16, 32]:

    print(f"Running hidden channels = {H}...")

    result = run_capacity_experiment(
        hidden_channels=H,
        seed=2026
    )

    capacity_results.append(result)

    print(
        f"Parameters = {result['Parameters']:,} | "
        f"Best epoch = {result['Best epoch']} | "
        f"log score = "
        f"{result['Validation log score']:.6f}"
    )

In [ ]:
capacity_df = pd.DataFrame([
    {
        "Hidden channels":
            r["Hidden channels"],
        "Parameters":
            r["Parameters"],
        "Best epoch":
            r["Best epoch"],
        "Validation NLL":
            r["Validation NLL"],
        "Validation log score":
            r["Validation log score"]
    }
    for r in capacity_results
])

capacity_df = (
    capacity_df
    .sort_values(
        "Validation log score",
        ascending=False
    )
    .reset_index(drop=True)
)

capacity_df

In [ ]:
result_4 = run_capacity_experiment(
    hidden_channels=4,
    seed=2026
)

print(
    f"Parameters = {result_4['Parameters']:,} | "
    f"Best epoch = {result_4['Best epoch']} | "
    f"log score = "
    f"{result_4['Validation log score']:.6f}"
)

In [ ]:
capacity_seed_results = []

for H in [8, 16]:
    for seed in [2027, 2028]:

        print(
            f"Running H={H}, seed={seed}..."
        )

        set_seed(seed)

        train_loader_H, val_loader_H = (
            make_loaders_for_history(
                history_length=21,
                seed=seed,
                batch_size=8
            )
        )

        model_H = EarthquakeConvLSTM(
            input_channels=1,
            hidden_channels=H,
            kernel_size=3,
            initial_mean_rate=train_mean_target
        ).to(device)

        (
            model_H,
            history_H,
            best_epoch_H,
            best_val_nll_H
        ) = fit_model(
            model=model_H,
            train_loader=train_loader_H,
            val_loader=val_loader_H,
            mask=ca_mask_tensor,
            device=device,
            learning_rate=1e-3,
            max_epochs=500,
            patience=20,
            verbose=False
        )

        result = {
            "Hidden channels": H,
            "Seed": seed,
            "Best epoch": best_epoch_H,
            "Validation log score": -best_val_nll_H
        }

        capacity_seed_results.append(result)

        print(
            f"Best epoch = {best_epoch_H}, "
            f"log score = {-best_val_nll_H:.6f}"
        )

In [ ]:
extra_capacity_results = []

for H in [8, 16]:

    for seed in [2029, 2030]:

        print(
            f"Running H={H}, seed={seed}..."
        )

        set_seed(seed)

        train_loader_H, val_loader_H = (
            make_loaders_for_history(
                history_length=21,
                seed=seed,
                batch_size=8
            )
        )

        model_H = EarthquakeConvLSTM(
            input_channels=1,
            hidden_channels=H,
            kernel_size=3,
            initial_mean_rate=train_mean_target
        ).to(device)

        (
            model_H,
            history_H,
            best_epoch_H,
            best_val_nll_H
        ) = fit_model(
            model=model_H,
            train_loader=train_loader_H,
            val_loader=val_loader_H,
            mask=ca_mask_tensor,
            device=device,
            learning_rate=1e-3,
            max_epochs=500,
            patience=20,
            verbose=False
        )

        result = {
            "Hidden channels": H,
            "Seed": seed,
            "Best epoch": best_epoch_H,
            "Validation log score":
                -best_val_nll_H
        }

        extra_capacity_results.append(
            result
        )

        print(
            f"Best epoch = {best_epoch_H}, "
            f"log score = "
            f"{-best_val_nll_H:.6f}"
        )

In [ ]:
capacity_all = pd.DataFrame([
    {
        "Hidden channels": 8,
        "Seed": 2026,
        "Validation log score": -0.038964
    },
    {
        "Hidden channels": 8,
        "Seed": 2027,
        "Validation log score": -0.037662
    },
    {
        "Hidden channels": 8,
        "Seed": 2028,
        "Validation log score": -0.040280
    },
    {
        "Hidden channels": 16,
        "Seed": 2026,
        "Validation log score": -0.039071
    },
    {
        "Hidden channels": 16,
        "Seed": 2027,
        "Validation log score": -0.039586
    },
    {
        "Hidden channels": 16,
        "Seed": 2028,
        "Validation log score": -0.040186
    },
] + extra_capacity_results)

capacity_summary = (
    capacity_all
    .groupby("Hidden channels")
    ["Validation log score"]
    .agg(
        mean="mean",
        std="std",
        median="median",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

capacity_summary

In [ ]:
class EarthquakeConvLSTMStacked(nn.Module):
    def __init__(
        self,
        input_channels=1,
        hidden_channels=8,
        kernel_size=3,
        num_layers=1,
        initial_mean_rate=None
    ):
        super().__init__()

        if num_layers not in [1, 2]:
            raise ValueError(
                "For this experiment, num_layers must be 1 or 2."
            )

        self.hidden_channels = hidden_channels
        self.num_layers = num_layers

        self.layers = nn.ModuleList()

        for layer_idx in range(num_layers):

            in_channels = (
                input_channels
                if layer_idx == 0
                else hidden_channels
            )

            self.layers.append(
                ConvLSTMCell(
                    input_channels=in_channels,
                    hidden_channels=hidden_channels,
                    kernel_size=kernel_size
                )
            )

        self.output_head = nn.Conv2d(
            in_channels=hidden_channels,
            out_channels=1,
            kernel_size=1
        )

        if initial_mean_rate is not None:

            mean_rate = float(
                initial_mean_rate
            )

            initial_bias = np.log(
                np.expm1(mean_rate)
            )

            nn.init.constant_(
                self.output_head.bias,
                initial_bias
            )

    def forward(self, x):

        # x:
        # (B, L, C, H, W)

        B, L, C, H, W = x.shape

        states_h = []
        states_c = []

        for _ in range(self.num_layers):

            h = torch.zeros(
                B,
                self.hidden_channels,
                H,
                W,
                device=x.device,
                dtype=x.dtype
            )

            c = torch.zeros_like(h)

            states_h.append(h)
            states_c.append(c)

        # ----------------------------------------------------
        # Process sequence one day at a time
        # ----------------------------------------------------

        for t in range(L):

            layer_input = x[:, t]

            for layer_idx, cell in enumerate(
                self.layers
            ):

                h, c = cell(
                    layer_input,
                    states_h[layer_idx],
                    states_c[layer_idx]
                )

                states_h[layer_idx] = h
                states_c[layer_idx] = c

                # Hidden state becomes input to next layer
                layer_input = h

        final_hidden = states_h[-1]

        raw_output = self.output_head(
            final_hidden
        )

        rate = (
            F.softplus(raw_output)
            + 1e-8
        )

        return rate

In [ ]:
for n_layers in [1, 2]:

    model_tmp = EarthquakeConvLSTMStacked(
        input_channels=1,
        hidden_channels=8,
        kernel_size=3,
        num_layers=n_layers,
        initial_mean_rate=train_mean_target
    )

    n_params = sum(
        p.numel()
        for p in model_tmp.parameters()
        if p.requires_grad
    )

    print(
        f"{n_layers} layer(s): "
        f"{n_params:,} parameters"
    )

In [ ]:
def run_depth_experiment(
    num_layers,
    seed=2026
):
    set_seed(seed)

    train_loader_D, val_loader_D = (
        make_loaders_for_history(
            history_length=21,
            seed=seed,
            batch_size=8
        )
    )

    model_D = EarthquakeConvLSTMStacked(
        input_channels=1,
        hidden_channels=8,
        kernel_size=3,
        num_layers=num_layers,
        initial_mean_rate=train_mean_target
    ).to(device)

    n_params = sum(
        p.numel()
        for p in model_D.parameters()
        if p.requires_grad
    )

    (
        model_D,
        history_D,
        best_epoch_D,
        best_val_nll_D
    ) = fit_model(
        model=model_D,
        train_loader=train_loader_D,
        val_loader=val_loader_D,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=500,
        patience=20,
        verbose=False
    )

    return {
        "Layers": num_layers,
        "Parameters": n_params,
        "Best epoch": best_epoch_D,
        "Validation log score":
            -best_val_nll_D,
        "model": model_D,
        "history": history_D
    }

In [ ]:
depth_results = []

for D in [1, 2]:

    print(
        f"Running ConvLSTM layers = {D}..."
    )

    result = run_depth_experiment(
        num_layers=D,
        seed=2026
    )

    depth_results.append(result)

    print(
        f"Parameters = {result['Parameters']:,} | "
        f"Best epoch = {result['Best epoch']} | "
        f"log score = "
        f"{result['Validation log score']:.6f}"
    )

In [ ]:
# ============================================================
# Reconstruct the locked Long-Term (LT) spatial forecast
# ============================================================

LT_TRAIN_START = pd.Timestamp("2010-01-01")
LT_TRAIN_END = pd.Timestamp("2019-12-31")

LT_EPSILON = 0.10
G = len(cell_ids)   # 1080

# ------------------------------------------------------------
# Training-calendar target-event counts
# M_w* >= 3.0
# ------------------------------------------------------------

lt_train_counts = daily_counts_target.loc[
    LT_TRAIN_START:LT_TRAIN_END
]

T_lt = len(lt_train_counts)

# Number of target earthquakes in each cell
N_g = (
    lt_train_counts
    .sum(axis=0)
    .to_numpy(dtype=np.float64)
)

# Statewide total
N_total = N_g.sum()

# Statewide mean daily rate
R_hat = N_total / T_lt

# ------------------------------------------------------------
# Raw spatial proportions
# ------------------------------------------------------------

p_hat = N_g / N_total

# ------------------------------------------------------------
# Uniform mixture already selected in the LT model
# ------------------------------------------------------------

p_tilde = (
    (1.0 - LT_EPSILON) * p_hat
    + LT_EPSILON / G
)

# ------------------------------------------------------------
# Expected count in each cell over the next 7 days
# ------------------------------------------------------------

lambda_lt = (
    FORECAST_HORIZON
    * R_hat
    * p_tilde
)

print("LT training days:", T_lt)
print("LT training target events:", int(N_total))
print("Daily statewide rate:", R_hat)
print(
    "Statewide 7-day expected count:",
    lambda_lt.sum()
)

print("\nCell-level LT rate:")
print("Minimum:", lambda_lt.min())
print("Maximum:", lambda_lt.max())
print("Mean:", lambda_lt.mean())

print("\nSpatial probabilities:")
print("Raw p sum:", p_hat.sum())
print("Smoothed p sum:", p_tilde.sum())
print(
    "Number of zero LT rates:",
    np.sum(lambda_lt == 0)
)

In [ ]:
# ============================================================
# Convert LT vector to 53 x 55 spatial map
# ============================================================

lt_map = np.zeros(
    (N_LAT, N_LON),
    dtype=np.float32
)

lt_map[
    lat_idx,
    lon_idx
] = lambda_lt.astype(np.float32)

print("LT map shape:", lt_map.shape)
print(
    "Non-zero outside California:",
    np.count_nonzero(
        lt_map[~ca_mask]
    )
)

print(
    "Statewide LT total from map:",
    lt_map.sum()
)

print(
    "Exact retained-cell reconstruction:",
    np.allclose(
        lt_map[lat_idx, lon_idx],
        lambda_lt
    )
)

assert lt_map.shape == (53, 55)

assert np.count_nonzero(
    lt_map[~ca_mask]
) == 0

assert np.allclose(
    lt_map[lat_idx, lon_idx],
    lambda_lt
)

In [ ]:
# ============================================================
# Log-transformed LT map for neural-network input
# ============================================================

lt_map_log = np.log1p(
    lt_map
).astype(np.float32)

print("Log LT map shape:", lt_map_log.shape)
print("Minimum:", lt_map_log.min())
print("Maximum:", lt_map_log.max())

print(
    "Non-zero outside California:",
    np.count_nonzero(
        lt_map_log[~ca_mask]
    )
)

assert np.count_nonzero(
    lt_map_log[~ca_mask]
) == 0

In [ ]:
print(
    "Expected statewide 7-day LT count:",
    7 * (4565 / 3652)
)

print(
    "Reconstructed statewide total:",
    lambda_lt.sum()
)

In [ ]:
# ============================================================
# Static spatial features
# ============================================================

mask_feature_map = (
    ca_mask.astype(np.float32)
)

lt_feature_map = (
    lt_map_log.astype(np.float32)
)

print(
    "Mask feature:",
    mask_feature_map.shape,
    mask_feature_map.min(),
    mask_feature_map.max()
)

print(
    "LT feature:",
    lt_feature_map.shape,
    lt_feature_map.min(),
    lt_feature_map.max()
)

In [ ]:
class EarthquakeConvLSTMWithStatic(nn.Module):

    def __init__(
        self,
        static_map,
        input_channels=1,
        hidden_channels=8,
        kernel_size=3,
        initial_mean_rate=None
    ):
        super().__init__()

        self.hidden_channels = hidden_channels

        self.convlstm = ConvLSTMCell(
            input_channels=input_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size
        )

        # ----------------------------------------------------
        # Static feature map
        # Stored with the model and automatically moved
        # between CPU / GPU.
        # ----------------------------------------------------

        static_tensor = torch.as_tensor(
            static_map,
            dtype=torch.float32
        ).unsqueeze(0).unsqueeze(0)

        self.register_buffer(
            "static_map",
            static_tensor
        )

        # 8 ConvLSTM channels + 1 static channel
        self.output_head = nn.Conv2d(
            in_channels=hidden_channels + 1,
            out_channels=1,
            kernel_size=1
        )

        # ----------------------------------------------------
        # Same output-bias initialisation as before
        # ----------------------------------------------------

        if initial_mean_rate is not None:

            mean_rate = float(
                initial_mean_rate
            )

            initial_bias = np.log(
                np.expm1(mean_rate)
            )

            nn.init.constant_(
                self.output_head.bias,
                initial_bias
            )

    def forward(self, x):

        B, L, C, H, W = x.shape

        h = torch.zeros(
            B,
            self.hidden_channels,
            H,
            W,
            device=x.device,
            dtype=x.dtype
        )

        c = torch.zeros_like(h)

        for t in range(L):

            h, c = self.convlstm(
                x[:, t],
                h,
                c
            )

        # Expand same static map across batch
        static = self.static_map.expand(
            B, -1, -1, -1
        )

        combined = torch.cat(
            [h, static],
            dim=1
        )

        raw_output = self.output_head(
            combined
        )

        rate = (
            F.softplus(raw_output)
            + 1e-8
        )

        return rate

In [ ]:
for name, static_map in [
    ("Mask", mask_feature_map),
    ("LT", lt_feature_map)
]:

    model_tmp = (
        EarthquakeConvLSTMWithStatic(
            static_map=static_map,
            input_channels=1,
            hidden_channels=8,
            kernel_size=3,
            initial_mean_rate=train_mean_target
        )
        .to(device)
    )

    n_params = sum(
        p.numel()
        for p in model_tmp.parameters()
        if p.requires_grad
    )

    with torch.no_grad():
        pred_tmp = model_tmp(xb)

    print(f"\n{name} model")
    print("Parameters:", f"{n_params:,}")
    print("Prediction shape:", pred_tmp.shape)
    print(
        "Finite:",
        torch.isfinite(pred_tmp).all().item()
    )
    print(
        "Positive:",
        (pred_tmp > 0).all().item()
    )

In [ ]:
def run_static_experiment(
    static_name,
    static_map,
    seed
):
    set_seed(seed)

    train_loader_S, val_loader_S = (
        make_loaders_for_history(
            history_length=21,
            seed=seed,
            batch_size=8
        )
    )

    model_S = EarthquakeConvLSTMWithStatic(
        static_map=static_map,
        input_channels=1,
        hidden_channels=8,
        kernel_size=3,
        initial_mean_rate=train_mean_target
    ).to(device)

    (
        model_S,
        history_S,
        best_epoch_S,
        best_val_nll_S
    ) = fit_model(
        model=model_S,
        train_loader=train_loader_S,
        val_loader=val_loader_S,
        mask=ca_mask_tensor,
        device=device,
        learning_rate=1e-3,
        max_epochs=500,
        patience=20,
        verbose=False
    )

    return {
        "Static feature": static_name,
        "Seed": seed,
        "Best epoch": best_epoch_S,
        "Validation NLL": best_val_nll_S,
        "Validation log score":
            -best_val_nll_S
    }

In [ ]:
static_results = []

for name, static_map in [
    ("Mask", mask_feature_map),
    ("LT", lt_feature_map)
]:

    for seed in [2026, 2027, 2028]:

        print(
            f"Running {name}, "
            f"seed={seed}..."
        )

        result = run_static_experiment(
            static_name=name,
            static_map=static_map,
            seed=seed
        )

        static_results.append(result)

        print(
            f"Best epoch = "
            f"{result['Best epoch']} | "
            f"log score = "
            f"{result['Validation log score']:.6f}"
        )

In [ ]:
static_results_df = pd.DataFrame(
    static_results
)

static_summary = (
    static_results_df
    .groupby("Static feature")
    ["Validation log score"]
    .agg(
        mean="mean",
        std="std",
        median="median",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

static_summary

In [ ]:
# ============================================================
# Freeze final ConvLSTM specification
# ============================================================

FINAL_HISTORY = 21
FINAL_HIDDEN = 8
FINAL_KERNEL = 3
FINAL_LAYERS = 1
FINAL_SEED = 2026

# H=8, seed=2026 model from the capacity experiment
final_model = capacity_results[0]["model"]

final_val_score = capacity_results[0][
    "Validation log score"
]

final_best_epoch = capacity_results[0][
    "Best epoch"
]

print("Final ConvLSTM specification")
print("----------------------------")
print("History length:", FINAL_HISTORY)
print("Hidden channels:", FINAL_HIDDEN)
print("Kernel size:", FINAL_KERNEL)
print("ConvLSTM layers:", FINAL_LAYERS)
print("Seed:", FINAL_SEED)
print("Best epoch:", final_best_epoch)
print(
    "Validation log score:",
    final_val_score
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in final_model.parameters()
        if p.requires_grad
    )
)

In [ ]:
# ============================================================
# Save frozen final ConvLSTM checkpoint
# ============================================================

checkpoint_path = MODEL_DIR / "final_convlstm_dynamic_only_seed2026.pt"

torch.save(
    {
        # Model weights
        "model_state_dict":
            final_model.state_dict(),

        # Frozen architecture
        "history_length":
            FINAL_HISTORY,
        "input_channels":
            1,
        "hidden_channels":
            FINAL_HIDDEN,
        "kernel_size":
            FINAL_KERNEL,
        "num_layers":
            FINAL_LAYERS,

        # Input / output specification
        "input_transform":
            "log1p",
        "input_magnitude_threshold":
            2.5,
        "target_magnitude_threshold":
            3.0,
        "forecast_horizon_days":
            7,
        "output_activation":
            "softplus",
        "loss":
            "masked_exact_poisson_nll",

        # Training specification
        "optimizer":
            "Adam",
        "learning_rate":
            1e-3,
        "adam_beta1":
            0.9,
        "adam_beta2":
            0.99,
        "batch_size":
            8,
        "early_stopping_patience":
            20,
        "seed":
            FINAL_SEED,

        # Selection information
        "best_epoch":
            final_best_epoch,
        "validation_log_score":
            final_val_score,

        # Temporal split
        "train_start":
            "2010-01-30",
        "train_end":
            "2019-12-24",
        "validation_start":
            "2020-01-01",
        "validation_end":
            "2022-12-24",
        "test_start":
            "2023-01-01",
        "test_end":
            "2025-12-24",

        # Spatial domain
        "grid_shape":
            (53, 55),
        "n_retained_cells":
            1080,

        # Important final model choice
        "static_mask_predictor":
            False,
        "static_lt_predictor":
            False
    },
    checkpoint_path
)

print(
    "Saved checkpoint:",
    checkpoint_path
)

In [ ]:
# ============================================================
# Verify checkpoint
# ============================================================

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

reloaded_model = EarthquakeConvLSTM(
    input_channels=
        checkpoint["input_channels"],
    hidden_channels=
        checkpoint["hidden_channels"],
    kernel_size=
        checkpoint["kernel_size"],
    initial_mean_rate=
        train_mean_target
).to(device)

reloaded_model.load_state_dict(
    checkpoint["model_state_dict"]
)

reloaded_model.eval()

print(
    "Checkpoint loaded successfully."
)

print(
    "Saved validation log score:",
    checkpoint["validation_log_score"]
)

In [ ]:
final_model.eval()

with torch.no_grad():

    pred_original = final_model(
        xb.to(device)
    )

    pred_reloaded = reloaded_model(
        xb.to(device)
    )

max_difference = (
    pred_original
    - pred_reloaded
).abs().max().item()

print(
    "Maximum prediction difference:",
    max_difference
)

print(
    "Predictions identical:",
    torch.allclose(
        pred_original,
        pred_reloaded,
        atol=1e-7,
        rtol=1e-6
    )
)

In [ ]:
# ============================================================
# Final test-period setup
# ============================================================

FINAL_HISTORY = 21

TEST_START = pd.Timestamp("2023-01-01")
TEST_END = pd.Timestamp("2025-12-24")

# Daily map dates
daily_dates = pd.DatetimeIndex(
    daily_counts_25.index
)

# Target / forecast-origin dates
target_origin_dates = pd.DatetimeIndex(
    future7_target.index
)

test_origins = target_origin_dates[
    (target_origin_dates >= TEST_START)
    & (target_origin_dates <= TEST_END)
]

print("Number of test origins:", len(test_origins))
print("First test origin:", test_origins[0])
print("Last test origin:", test_origins[-1])

assert len(test_origins) == 1089
assert test_origins[0] == TEST_START
assert test_origins[-1] == TEST_END

In [ ]:
from torch.utils.data import Dataset, DataLoader


class FinalTestSequenceDataset(Dataset):

    def __init__(
        self,
        input_maps,
        target_maps,
        daily_dates,
        target_origin_dates,
        origins,
        history_length
    ):

        self.input_maps = input_maps
        self.target_maps = target_maps

        self.daily_dates = pd.DatetimeIndex(
            daily_dates
        )

        self.target_origin_dates = (
            pd.DatetimeIndex(
                target_origin_dates
            )
        )

        self.origins = pd.DatetimeIndex(
            origins
        )

        self.history_length = (
            history_length
        )

        # Date -> array-position lookup
        self.daily_lookup = {
            date: i
            for i, date
            in enumerate(self.daily_dates)
        }

        self.target_lookup = {
            date: i
            for i, date
            in enumerate(
                self.target_origin_dates
            )
        }

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):

        origin = self.origins[idx]

        input_end = (
            self.daily_lookup[origin]
        )

        input_start = (
            input_end
            - self.history_length
            + 1
        )

        target_pos = (
            self.target_lookup[origin]
        )

        # Shape:
        # (L, 53, 55)
        x = self.input_maps[
            input_start:
            input_end + 1
        ]

        # Add channel dimension:
        # (L, 1, 53, 55)
        x = torch.as_tensor(
            x,
            dtype=torch.float32
        ).unsqueeze(1)

        # Target:
        # (53, 55) -> (1, 53, 55)
        y = torch.as_tensor(
            self.target_maps[
                target_pos
            ],
            dtype=torch.float32
        ).unsqueeze(0)

        return x, y

In [ ]:
test_dataset = FinalTestSequenceDataset(
    input_maps=daily_maps_25_log,
    target_maps=future7_target_maps,
    daily_dates=daily_dates,
    target_origin_dates=target_origin_dates,
    origins=test_origins,
    history_length=FINAL_HISTORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(
    "Test samples:",
    len(test_dataset)
)

In [ ]:
x_test0, y_test0 = (
    test_dataset[0]
)

print(
    "Input shape:",
    x_test0.shape
)

print(
    "Target shape:",
    y_test0.shape
)

print(
    "Input start date:",
    TEST_START
    - pd.Timedelta(
        days=FINAL_HISTORY - 1
    )
)

print(
    "Forecast origin:",
    TEST_START
)

print(
    "Target period:",
    TEST_START
    + pd.Timedelta(days=1),
    "to",
    TEST_START
    + pd.Timedelta(days=7)
)

print(
    "Input earthquake total:",
    np.expm1(
        x_test0.numpy()
    ).sum()
)

print(
    "Target earthquake total:",
    y_test0.sum().item()
)

In [ ]:
xb_test, yb_test = next(
    iter(test_loader)
)

print(
    "Test batch input:",
    xb_test.shape
)

print(
    "Test batch target:",
    yb_test.shape
)

assert xb_test.shape[1:] == (
    21, 1, 53, 55
)

assert yb_test.shape[1:] == (
    1, 53, 55
)

In [ ]:
import inspect
print(inspect.signature(evaluate_model))

In [ ]:
reloaded_model.eval()

test_nll, test_log_score = evaluate_model(
    model=reloaded_model,
    data_loader=test_loader,
    mask=ca_mask_tensor,
    device=device
)

print("Final ConvLSTM test NLL:", test_nll)
print("Final ConvLSTM test log score:", test_log_score)

In [ ]:
# ============================================================
# Generate and save final ConvLSTM test predictions
# ============================================================

reloaded_model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(
            device,
            non_blocking=True
        )

        pred = reloaded_model(xb)

        # Keep only the 1080 retained California cells
        pred_cells = pred[
            :, 0, lat_idx, lon_idx
        ]

        target_cells = yb[
            :, 0, lat_idx, lon_idx
        ]

        all_predictions.append(
            pred_cells.cpu().numpy()
        )

        all_targets.append(
            target_cells.numpy()
        )

convlstm_test_pred = np.concatenate(
    all_predictions,
    axis=0
)

convlstm_test_y = np.concatenate(
    all_targets,
    axis=0
)

print(
    "Prediction shape:",
    convlstm_test_pred.shape
)

print(
    "Target shape:",
    convlstm_test_y.shape
)

assert convlstm_test_pred.shape == (
    1089, 1080
)

assert convlstm_test_y.shape == (
    1089, 1080
)

assert np.isfinite(
    convlstm_test_pred
).all()

assert (
    convlstm_test_pred > 0
).all()

In [ ]:
print(
    "Observed mean count per day-cell:",
    convlstm_test_y.mean()
)

print(
    "Predicted mean count per day-cell:",
    convlstm_test_pred.mean()
)

print(
    "Observed mean statewide 7-day count:",
    convlstm_test_y.sum(axis=1).mean()
)

print(
    "Predicted mean statewide 7-day count:",
    convlstm_test_pred.sum(axis=1).mean()
)

print(
    "Observed total target count:",
    convlstm_test_y.sum()
)

print(
    "Zero fraction:",
    np.mean(
        convlstm_test_y == 0
    )
)

print(
    "Maximum observed cell target:",
    convlstm_test_y.max()
)

print(
    "Prediction range:",
    convlstm_test_pred.min(),
    convlstm_test_pred.max()
)

In [ ]:
# ============================================================
# Independent check of the Poisson test log score
# ============================================================

lambda_test = np.maximum(
    convlstm_test_pred.astype(np.float64),
    1e-12
)

y_test = convlstm_test_y.astype(
    np.float64
)

poisson_nll_cells = (
    lambda_test
    - y_test * np.log(lambda_test)
    + torch.lgamma(
        torch.from_numpy(
            y_test + 1.0
        )
    ).numpy()
)

manual_test_nll = (
    poisson_nll_cells.mean()
)

manual_test_log_score = (
    -manual_test_nll
)

print(
    "Manual test NLL:",
    manual_test_nll
)

print(
    "Manual test log score:",
    manual_test_log_score
)

print(
    "Difference from evaluate_model:",
    manual_test_log_score
    - test_log_score
)

In [ ]:
# ============================================================
# Save final ConvLSTM test outputs
# ============================================================

np.savez_compressed(
    PRED_DIR / "final_convlstm_test_predictions.npz",
    predictions=convlstm_test_pred,
    targets=convlstm_test_y,
    origins=test_origins.values,
    cell_ids=np.asarray(cell_ids)
)

print(
    "Saved:",
    PRED_DIR / "final_convlstm_test_predictions.npz"
)